# F02 Vector Geometry, Features, and Probes


## What this foundation lab is for

This lab exists to reduce the most common beginner failure modes before the article-first path starts.


## Skills you should leave with

- Understand features as directions, projections, and separability rather than fuzzy semantic labels.
- Distinguish similar directions, entangled directions, and directions that a probe can read out.
- Explain in geometric language why a probe can read out a property.


## Ship these outputs

- One diagram of directions, projections, and a decision boundary.
- One short probe-weight interpretation.
- One failure note about geometric entanglement.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Jonny-English/circuits-zoom-in-fresh-20260325.git"
REPO_DIR = "circuits-zoom-in-fresh-20260325"
REPO_BRANCH = "main"

if "google.colab" in sys.modules:
    candidate = Path("/content") / REPO_DIR
    if not candidate.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(candidate)],
            check=True,
        )
    os.chdir(candidate)

root = Path.cwd().resolve()
while not (root / "content" / "course.json").exists():
    if root.parent == root:
        raise RuntimeError("Run this notebook from the repository root.")
    root = root.parent

import matplotlib.pyplot as plt
import numpy as np

feature_truth = np.array([1.0, 0.25])
feature_spurious = np.array([0.65, 0.76])
feature_truth = feature_truth / np.linalg.norm(feature_truth)
feature_spurious = feature_spurious / np.linalg.norm(feature_spurious)

rng = np.random.default_rng(5)
positive = rng.normal(loc=[1.2, 0.5], scale=[0.22, 0.18], size=(18, 2))
negative = rng.normal(loc=[-0.9, -0.2], scale=[0.25, 0.2], size=(18, 2))
points = np.vstack([positive, negative])
labels = np.concatenate([np.ones(len(positive)), np.zeros(len(negative))])

centered_labels = labels * 2 - 1
probe = np.linalg.lstsq(points, centered_labels, rcond=None)[0]
probe = probe / np.linalg.norm(probe)

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(positive[:, 0], positive[:, 1], color="#1f5f8b", label="positive")
ax.scatter(negative[:, 0], negative[:, 1], color="#c96a28", label="negative")
for vector, color, label in [
    (feature_truth, "#0f3d5e", "feature_truth"),
    (feature_spurious, "#8b5e34", "feature_spurious"),
    (probe, "#2d6a4f", "probe"),
]:
    ax.arrow(0, 0, vector[0], vector[1], width=0.02, color=color, length_includes_head=True)
    ax.text(vector[0] * 1.08, vector[1] * 1.08, label)
ax.axhline(0, color="0.85")
ax.axvline(0, color="0.85")
ax.set_title("Directions, projections, and a probe")
ax.legend()
ax.set_aspect("equal")
plt.tight_layout()

print("cos(feature_truth, probe) =", round(cosine(feature_truth, probe), 3))
print("cos(feature_spurious, probe) =", round(cosine(feature_spurious, probe), 3))
print("Average positive projection onto probe:", round(float((positive @ probe).mean()), 3))
print("Average negative projection onto probe:", round(float((negative @ probe).mean()), 3))


## Takeaway

Once you see features as directions, probes become geometry plus supervision rather than mysterious semantic magic.


## Self-check questions

- If two feature directions are strongly correlated, why can a high probe score still mislead you?
- When is cosine similarity useful, and when does it flatten structure too aggressively?
- Once features are treated as directions, how would you reinterpret monosemanticity in M02?
- If you cannot answer at least two questions without reopening the notebook, stay here before moving to the article track.
